### Testing HVAC_water_system

- Load: process_load
- System always 9 hours on
- Pump operation on load
- System control: Scheduled: Nov, Dec, Jan, Feb: Heating rest Cooling
- Losses: 20*(22-T_w) if m_w > 0 else 0.2*(22-T_w)


In [1]:
import opensimula as osm

dict = {
    "name": "Process water system",
    "time_step": 3600,
    "n_time_steps": 8760,
    "initial_time": "01/01/2001 00:00:00",
    "simulation_file_met": "Sevilla",
    "components": [
        {
            "type": "File_met",
            "name": "Sevilla",
            "file_type": "MET",
            "file_name": "../mets/sevilla.met"
        },
        {
            "type": "File_data",
            "name": "process_load",
            "file_type": "CSV",
            "file_name": "process_load.csv",
            "file_step": "SIMULATION"
        },
        {
            "type": "Day_schedule",
            "name": "working_day",
            "time_steps": [8*3600, 9*3600],
            "values": [0, 1, 0],
            "interpolation": "STEP",
        },
        {
            "type": "Day_schedule",
            "name": "off_day",
            "time_steps": [],
            "values": [0],
            "interpolation": "STEP",
        },
        {
            "type": "Day_schedule",
            "name": "heating_day",
            "time_steps": [],
            "values": [1],
            "interpolation": "STEP",
        },
        {
            "type": "Day_schedule",
            "name": "cooling_day",
            "time_steps": [],
            "values": [-1],
            "interpolation": "STEP",
        },
        {
            "type": "Week_schedule",
            "name": "working_week",
            "days_schedules": [
                "working_day"
            ],
        },
        {
            "type": "Week_schedule",
            "name": "off_week",
            "days_schedules": [
                "off_day"
            ],
        },
        {
            "type": "Week_schedule",
            "name": "heating_week",
            "days_schedules": [
                "heating_day"
            ],
        },
        {
            "type": "Week_schedule",
            "name": "cooling_week",
            "days_schedules": [
                "cooling_day"
            ],
        },
        {
            "type": "Year_schedule",
            "name": "on_schedule",
            "periods": [],
            "weeks_schedules": ["working_week"],
        },
        {
            "type": "Year_schedule",
            "name": "mode_schedule",
            "periods": ["01/03","01/11"],
            "weeks_schedules": ["heating_week", "cooling_week", "heating_week"],
        },
        {
            "type":"Pump",
            "name":"pump",
            "nominal_water_flow": 0.4137,
            "nominal_pressure": 100000,
            "nominal_power": 70,
        },
        {
            "type":"Chiller_heat_pump",
            "name":"heat_pump",
            "chiller_type":"CHILLER_HEAT_PUMP",
            "nominal_cooling_capacity": 8000,
            "nominal_cooling_power": 4000,
            "nominal_heating_capacity": 9000,
            "nominal_heating_power": 4500,
            "nominal_water_flow": 0.4137
        },
        {
            "type":"HVAC_water_system",
            "name":"water_system",
            "water_thermal_generator": "heat_pump",
            "pump": "pump",
            "design_water_flow": 0.4137,
            "heating_water_setpoint": "50",
            "cooling_water_setpoint": "7",
            "total_water_volume": 0.1,
            "system_on_off":"g",
            "pump_operation": "ON_LOAD",
            "system_control": "SCHEDULE_CONTROL",
            "system_mode":"f",
            "input_variables":["Q = process_load.Q","f = mode_schedule.values","g= on_schedule.values"],
            "Q_process":"Q",
            "Q_loss":" 20*(22-T_w) if m_w > 0 else 0.2*(22-T_w)",
        }
    ]
}

sim = osm.Simulation()

pro = sim.new_project("pro")
pro.read_dict(dict)

Reading project data from dictonary
Reading completed.
Checking project: Process water system
Checking completed.


In [2]:
pro.simulate()

Calculating solar direct shadows ...
Simulating Process water system: ...


100%|██████████| 8760/8760 [00:05<00:00, 1526.57step/s, n_iter=3]


In [12]:
T_wgo = pro.component("water_system").variable("T_WGO")
T_wgi = pro.component("water_system").variable("T_WGI")
T_wco = pro.component("water_system").variable("T_WCO")
T_wci= pro.component("water_system").variable("T_WCI")
f_load = pro.component("water_system").variable("generator_part_load")
Q_process = pro.component("water_system").variable("Q_process")
Q_loss = pro.component("water_system").variable("Q_loss")
Q_pump = pro.component("water_system").variable("Q_pump")
Q_gen = pro.component("water_system").variable("Q_gen")
delta_U = pro.component("water_system").variable("delta_U")
eff = pro.component("water_system").variable("generator_efficiency")
fcp = pro.component("water_system").variable("generator_part_load")


In [4]:
sim.plot(pro.dates(),[T_wgo,T_wgi,T_wco,T_wci])

In [13]:
sim.plot(pro.dates(),[eff,fcp],axis=[1,2])

In [5]:
sim.plot(pro.dates(),[T_wgo,T_wgi,T_wco,T_wci],interval=["01/10/2001","01/14/2001"])

In [6]:
sim.plot(pro.dates(),[T_wgo,T_wgi,T_wco,T_wci],interval=["07/08/2001","07/12/2001"])

In [7]:
sim.plot(pro.dates(),[Q_process,Q_pump,Q_gen,Q_loss,delta_U],interval=["01/10/2001","01/12/2001"])

In [8]:
sim.plot(pro.dates(),[Q_process,Q_pump,Q_gen,Q_loss,delta_U],interval=["07/08/2001","07/10/2001"])

In [9]:
system_df = pro.component("water_system").variable_dataframe()
system_df

,date,on_off,mode,T_WGO,T_WGI,T_WCI,T_WCO,T_WAVG,water_flow,Q_gen,...,delta_U,pump_power,generator_power,generator_efficiency,generator_part_load,heating_water_setpoint,cooling_water_setpoint,Q,f,g
0,2001-01-01 00:30:00,0.0,1.0,20.003442,20.003442,20.003442,20.003442,20.003442,0.0,0.0,...,0.399429,0.0,0.0,0.0,0.0,50.0,7.0,-194.0,1.0,0.0
1,2001-01-01 01:30:00,0.0,1.0,20.006878,20.006878,20.006878,20.006878,20.006878,0.0,0.0,...,0.398742,0.0,0.0,0.0,0.0,50.0,7.0,-287.0,1.0,0.0
2,2001-01-01 02:30:00,0.0,1.0,20.010308,20.010308,20.010308,20.010308,20.010308,0.0,0.0,...,0.398056,0.0,0.0,0.0,0.0,50.0,7.0,-365.0,1.0,0.0
3,2001-01-01 03:30:00,0.0,1.0,20.013732,20.013732,20.013732,20.013732,20.013732,0.0,0.0,...,0.397371,0.0,0.0,0.0,0.0,50.0,7.0,-428.0,1.0,0.0
4,2001-01-01 04:30:00,0.0,1.0,20.017150,20.017150,20.017150,20.017150,20.017150,0.0,0.0,...,0.396687,0.0,0.0,0.0,0.0,50.0,7.0,-488.0,1.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8755,2001-12-31 19:30:00,0.0,1.0,49.519249,49.222098,49.358386,49.342004,49.360434,0.0,0.0,...,-5.473770,0.0,0.0,0.0,0.0,50.0,7.0,0.0,1.0,0.0
8756,2001-12-31 20:30:00,0.0,1.0,49.471637,49.174486,49.310775,49.294392,49.312823,0.0,0.0,...,-5.464244,0.0,0.0,0.0,0.0,50.0,7.0,0.0,1.0,0.0
8757,2001-12-31 21:30:00,0.0,1.0,49.424110,49.126959,49.263247,49.246865,49.265295,0.0,0.0,...,-5.454736,0.0,0.0,0.0,0.0,50.0,7.0,-239.0,1.0,0.0
8758,2001-12-31 22:30:00,0.0,1.0,49.376665,49.079514,49.215803,49.199421,49.217851,0.0,0.0,...,-5.445244,0.0,0.0,0.0,0.0,50.0,7.0,-654.0,1.0,0.0


In [10]:
import plotly.graph_objects as go

system_df = pro.component("water_system").variable_dataframe(
    frequency="monthly", value="sum", pos_neg_columns=["Q_gen", "Q_process","delta_U"]
)
cols = ["Q_gen_pos", "Q_gen_neg", "Q_process_pos", "Q_process_neg", "Q_pump", "Q_loss", "delta_U_pos", "delta_U_neg"]
months = system_df.index.strftime("%b")

fig = go.Figure()
for col in cols:
    fig.add_trace(go.Bar(x=months, y=system_df[col], name=col))

fig.update_layout(
    title="Water system monthly energy balance",
    xaxis_title="Month",
    yaxis_title="Energy (Wh)",
    barmode="group",
)
fig.show()

In [11]:
system_df_year = pro.component("water_system").variable_dataframe(
    frequency="yearly", value="sum", pos_neg_columns=["Q_gen", "Q_process","delta_U"]
)
print(f"Cooling load: {system_df_year['Q_process_pos'].values[0]/1e6}")
print(f"Heating load: {system_df_year['Q_process_neg'].values[0]/1e6}")
print(f"Pump heat: {system_df_year['Q_pump'].values[0]/1e6}")
print(f"Losses: {system_df_year['Q_loss'].values[0]/1e6}")
print(f"delta_U_pos: {system_df_year['delta_U_pos'].values[0]/1e6}")
print(f"delta_U_neg: {system_df_year['delta_U_neg'].values[0]/1e6}")
print(f"Cooling generation: {system_df_year['Q_gen_neg'].values[0]/1e6}")
print(f"Heating generation: {system_df_year['Q_gen_pos'].values[0]/1e6}")

Cooling load: 2.347613
Heating load: -0.120633
Pump heat: 0.12026
Losses: 0.34580827972224265
delta_U_pos: 0.0350031146665265
delta_U_neg: -0.03172249680782765
Cooling generation: -2.9200937413474226
Heating generation: 0.228625428945112
